# Lezione 7 · Embedding: il significato come geometria

La mappa degli ormeggi a due direzioni, la similarità coseno, il gioco degli abbinamenti, i conti del bibliotecario e lo schedario di bordo.

Ogni numero calcolato qui viene confrontato con quello scritto nel libro: un segno ✓ vuol dire che coincidono.

In [1]:
import math
import numpy as np

def controlla(nome, valore, atteso, tolleranza=0.01):
    """Confronta un valore calcolato con quello scritto nel libro."""
    ok = abs(valore - atteso) <= tolleranza
    segno = "✓" if ok else "✗ DIVERSO DAL LIBRO"
    print(f"{segno}  {nome}: calcolato {valore:.4g}, nel libro {atteso:g}")
    assert ok, nome

def conta(nome, valore, atteso):
    """Come controlla, per i numeri interi: devono coincidere esattamente."""
    ok = valore == atteso
    segno = "✓" if ok else "✗ DIVERSO DAL LIBRO"
    intero = lambda x: f"{x:,.0f}".replace(",", ".")
    print(f"{segno}  {nome}: calcolato {intero(valore)}, nel libro {intero(atteso)}")
    assert ok, nome

def sigmoide(z):
    return 1 / (1 + math.exp(-z))

def softmax(z):
    e = np.exp(np.array(z, dtype=float) - np.max(z))
    return e / e.sum()

## Dall'one-hot alla tabella

*Nel racconto: «barca» il 1023, «barbiere» il 1024; un vettore lungo quanto tutto il vocabolario.*

In [2]:
controlla("tabella con V = 50.000 e d = 768: pesi (milioni)", 50_000 * 768 / 1e6, 38.4)

# moltiplicare il one-hot per la tabella non fa altro che pescare la riga giusta
mappa = {"barca": (0.9, -0.8), "nave": (0.8, -0.9), "pesce": (0.8, 0.9), "trattore": (-0.9, -0.8), "gatto": (-0.8, 0.8)}
parole = list(mappa)
E = np.array([mappa[p] for p in parole])           # una riga per parola: terra → mare, oggetto → vivo
one_hot = np.eye(len(parole))[parole.index("pesce")]
assert np.array_equal(one_hot @ E, E[parole.index("pesce")])
print("one-hot di «pesce»:", one_hot, "→ la sua riga:", one_hot @ E)

✓  tabella con V = 50.000 e d = 768: pesi (milioni): calcolato 38.4, nel libro 38.4
one-hot di «pesce»: [0. 0. 1. 0. 0.] → la sua riga: [0.8 0.9]


## La similarità coseno

*Nel racconto: barca e nave «quasi gemelle»; pesce «esattamente a metà».*

In [3]:
lung = np.linalg.norm
coseno = lambda a, b: float(np.dot(a, b) / (lung(a) * lung(b)))
barca = np.array(mappa["barca"])
controlla("lunghezza di «barca»", lung(barca), 1.204, 0.001)
for parola, scalare, cos in [("nave", 1.44, 0.99), ("pesce", 0, 0), ("trattore", -0.17, -0.12), ("gatto", -1.36, -1.00)]:
    controlla(f"barca · {parola}", float(barca @ mappa[parola]), scalare)
    controlla(f"coseno tra barca e {parola}", coseno(barca, mappa[parola]), cos)
controlla("lunghezza di «gatto»", lung(mappa["gatto"]), 1.131, 0.001)
controlla("il coseno di gatto, con tre cifre", coseno(barca, mappa["gatto"]), -0.998, 0.0005)

✓  lunghezza di «barca»: calcolato 1.204, nel libro 1.204
✓  barca · nave: calcolato 1.44, nel libro 1.44
✓  coseno tra barca e nave: calcolato 0.9931, nel libro 0.99
✓  barca · pesce: calcolato 2.665e-17, nel libro 0
✓  coseno tra barca e pesce: calcolato 1.838e-17, nel libro 0
✓  barca · trattore: calcolato -0.17, nel libro -0.17
✓  coseno tra barca e trattore: calcolato -0.1172, nel libro -0.12
✓  barca · gatto: calcolato -1.36, nel libro -1.36
✓  coseno tra barca e gatto: calcolato -0.9983, nel libro -1
✓  lunghezza di «gatto»: calcolato 1.131, nel libro 1.131
✓  il coseno di gatto, con tre cifre: calcolato -0.9983, nel libro -0.998


## La compagnia delle parole

*Nel racconto: «dimmi con chi vai»; il gioco delle parole vicine; il trattore che fa un passo verso il mare.*

In [4]:
frase = "la barca entra nel porto".split()
i = frase.index("barca")
compagnia = [frase[j] for j in range(max(0, i - 2), min(len(frase), i + 3)) if j != i]
print("con una finestra di due parole, «barca» forma le coppie con:", compagnia)
assert compagnia == ["la", "entra", "nel"]

# il passo dalla terra al mare
arrivo = np.array(mappa["trattore"]) + np.array([1.8, 0])
vicina = max(parole, key=lambda p: coseno(arrivo, mappa[p]) if p != "trattore" else -2)
print("trattore + (1,8; 0) =", arrivo, "→ la parola più vicina è", vicina)
assert np.allclose(arrivo, mappa["barca"]) and vicina == "barca"

con una finestra di due parole, «barca» forma le coppie con: ['la', 'entra', 'nel']
trattore + (1,8; 0) = [ 0.9 -0.8] → la parola più vicina è barca


## Lo specchio

*Nel racconto: «uomo sta a programmatore come donna sta a...»; il gozzo vicino ai gommoni.*

La correzione proposta: togliere da ogni parola la sua componente lungo una direzione g di lunghezza 1.

In [5]:
g = np.array([0.6, 0.8])
v = np.array([0.9, -0.8])
v_corretta = v - (v @ g) * g
controlla("dopo la correzione, la componente lungo g", float(v_corretta @ g), 0, 1e-12)

✓  dopo la correzione, la componente lungo g: calcolato 5.684e-17, nel libro 0


## Pietre fisse e pietre che si muovono

*Nel racconto: le due pesche.*

In [6]:
mare, frutta = np.array([1.0, 0.0]), np.array([0.0, 1.0])
pesca = (mare + frutta) / 2
controlla("lunghezza del punto medio", lung(pesca), 0.707, 0.001)
controlla("coseno della «pesca» statica con il mare", coseno(pesca, mare), 0.71)
controlla("coseno della «pesca» statica con la frutta", coseno(pesca, frutta), 0.71)

✓  lunghezza del punto medio: calcolato 0.7071, nel libro 0.707
✓  coseno della «pesca» statica con il mare: calcolato 0.7071, nel libro 0.71
✓  coseno della «pesca» statica con la frutta: calcolato 0.7071, nel libro 0.71


## Imparare per contrasto

*Nel racconto: il gioco di abbinare foto e didascalie.*

In [7]:
s = np.array([0.9, 0.1, 0.2])                       # il primo è il passaggio giusto
for v, a in zip(np.exp(s), [2.46, 1.11, 1.22]):
    controlla("esponenziale", v, a)
controlla("somma", np.exp(s).sum(), 4.79)
p = softmax(s)[0]
controlla("τ = 1: probabilità del passaggio giusto", p, 0.51)
controlla("τ = 1: errore", -math.log(p), 0.67)
p = softmax(s / 0.1)[0]
controlla("τ = 0,1: probabilità", p, 0.999, 0.0005)
controlla("τ = 0,1: errore", -math.log(p), 0.001, 0.0005)

✓  esponenziale: calcolato 2.46, nel libro 2.46
✓  esponenziale: calcolato 1.105, nel libro 1.11
✓  esponenziale: calcolato 1.221, nel libro 1.22
✓  somma: calcolato 4.786, nel libro 4.79
✓  τ = 1: probabilità del passaggio giusto: calcolato 0.5139, nel libro 0.51
✓  τ = 1: errore: calcolato 0.6657, nel libro 0.67
✓  τ = 0,1: probabilità: calcolato 0.9988, nel libro 0.999
✓  τ = 0,1: errore: calcolato 0.001247, nel libro 0.001


## Lo stesso scafo

*Nel racconto: tre barche dallo stesso cantiere; il corpo e la testa.*

In [8]:
d, V, K = 768, 50_000, 3
controlla("testa per scrivere, d × V (milioni di pesi)", d * V / 1e6, 38.4)
conta("testa per decidere, con tre bandiere", d * K, 2304)

✓  testa per scrivere, d × V (milioni di pesi): calcolato 38.4, nel libro 38.4
✓  testa per decidere, con tre bandiere: calcolato 2.304, nel libro 2.304


## Tagliare con giudizio

*Nel racconto: l'atlante con le strisce ripetute ai bordi.*

In [9]:
N, L, o = 10_000, 500, 50
controlla("(N − L) / (L − o)", (N - L) / (L - o), 21.1, 0.05)
conta("pezzi", math.ceil((N - L) / (L - o)) + 1, 23)

# la prova: si taglia davvero, e si conta
partenze = list(range(0, N - L, L - o)) + [N - L]
assert len(partenze) == 23 and partenze[-1] + L == N

✓  (N − L) / (L − o): calcolato 21.11, nel libro 21.1
✓  pezzi: calcolato 23, nel libro 23


## Il conto dell'allievo e le matrioske

*Nel racconto: i milioni di pezzi da mille numeri ciascuno; la carta d'insieme e quella di dettaglio.*

In [10]:
pezzi, numeri, byte = 1_000_000, 1024, 4
controlla("memoria (miliardi di byte)", pezzi * numeri * byte / 1e9, 4.1, 0.05)
controlla("moltiplicazioni per domanda (miliardi)", pezzi * numeri / 1e9, 1, 0.05)
conta("con i primi 64 numeri: byte", pezzi * 64 * byte, 256_000_000)
controlla("con i primi 64 numeri: conti (un sedicesimo)", 64 / numeri, 1 / 16)
controlla("riordinare cento candidati con il vettore intero (migliaia di moltiplicazioni)", 100 * numeri / 1e3, 100, 3)

✓  memoria (miliardi di byte): calcolato 4.096, nel libro 4.1
✓  moltiplicazioni per domanda (miliardi): calcolato 1.024, nel libro 1
✓  con i primi 64 numeri: byte: calcolato 256.000.000, nel libro 256.000.000
✓  con i primi 64 numeri: conti (un sedicesimo): calcolato 0.0625, nel libro 0.0625
✓  riordinare cento candidati con il vettore intero (migliaia di moltiplicazioni): calcolato 102.4, nel libro 100


## L'archivio a moli

*Nel racconto: «prima cerca il molo giusto, poi guarda solo le barche ormeggiate lì».*

In [11]:
moli, visitati = 1000, 10
controlla("pezzi guardati (%)", visitati / moli * 100, 1)

✓  pezzi guardati (%): calcolato 1, nel libro 1


## Lo schedario di bordo

*Nel racconto: l'elenco dei fari; «si ristampa la sua scheda, non il manuale».*

In [12]:
controlla("P(Genova | porto di)", 120 / 1000, 0.12)
controlla("trigrammi possibili con 50.000 parole (× 10¹⁴)", 50_000 ** 3 / 1e14, 1.25)

✓  P(Genova | porto di): calcolato 0.12, nel libro 0.12
✓  trigrammi possibili con 50.000 parole (× 10¹⁴): calcolato 1.25, nel libro 1.25


## Kata per casa

**Esercizio 1.**

In [13]:
controlla("(1; 0) e (1; 1)", coseno([1, 0], [1, 1]), 0.71)
controlla("(1; 0) e (0; 1)", coseno([1, 0], [0, 1]), 0)
controlla("(1; 0) e (−1; 0)", coseno([1, 0], [-1, 0]), -1)

✓  (1; 0) e (1; 1): calcolato 0.7071, nel libro 0.71
✓  (1; 0) e (0; 1): calcolato 0, nel libro 0
✓  (1; 0) e (−1; 0): calcolato -1, nel libro -1


## Prova tu

Aggiungi parole alla mappa, scegliendo tu le coordinate, e cerca quale sta più vicina a ciascuna.

In [14]:
mappa["gozzo"] = (0.85, -0.6)
for p in mappa:
    altre = [q for q in mappa if q != p]
    print(f"{p:9s} → la più vicina è {max(altre, key=lambda q: coseno(mappa[p], mappa[q]))}")

barca     → la più vicina è gozzo
nave      → la più vicina è barca
pesce     → la più vicina è gozzo
trattore  → la più vicina è gatto
gatto     → la più vicina è trattore
gozzo     → la più vicina è barca
